# 02. Exploratory Data Analysis (EDA)

This notebook produces:
1. **Graph 1**: NSW RRP time series
2. **Graph 2**: RRP distribution histogram
3. **Graph 3**: Weekday-hour heatmap

Key questions:
- What is the overall price behavior?
- How heavy is the right tail (extreme prices)?
- Which hours/days have highest average prices?

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src import io as data_io
from src import features
from src import plots

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Processed Data

In [ ]:
# Load processed data
df = data_io.load_processed_data('nsw_processed')

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nBasic stats:")
display(df.describe())

## 2. Graph 1: RRP Time Series

**Purpose**: Overview of price behavior over the analysis period. Identify periods of high volatility.

In [ ]:
# Plot full time series
fig = plots.plot_rrp_timeseries(
    df,
    spike_threshold=config.SPIKE_ABSOLUTE_THRESHOLD,
    title=f"NSW Regional Reference Price (5-min intervals)"
)
plt.show()

In [ ]:
# Summary statistics for prices
print("RRP Summary Statistics:")
print(f"  Mean: ${df['rrp'].mean():.2f}/MWh")
print(f"  Median: ${df['rrp'].median():.2f}/MWh")
print(f"  Std Dev: ${df['rrp'].std():.2f}")
print(f"  Min: ${df['rrp'].min():.2f}/MWh")
print(f"  Max: ${df['rrp'].max():.2f}/MWh")
print(f"\nPercentiles:")
for p in [90, 95, 99, 99.5, 99.9]:
    val = df['rrp'].quantile(p/100)
    print(f"  {p}th: ${val:.2f}/MWh")

## 3. Graph 2: RRP Distribution

**Purpose**: Show price distribution shape, especially the heavy right tail.

In [ ]:
# Plot distribution
fig = plots.plot_rrp_distribution(
    df,
    percentiles=[90, 95, 99]
)
plt.show()

In [ ]:
# Analyze extreme values
threshold = config.SPIKE_ABSOLUTE_THRESHOLD
n_above_threshold = (df['rrp'] >= threshold).sum()
pct_above_threshold = n_above_threshold / len(df) * 100

print(f"\nIntervals with RRP >= ${threshold}/MWh:")
print(f"  Count: {n_above_threshold:,}")
print(f"  Percentage: {pct_above_threshold:.2f}%")

# Negative prices
n_negative = (df['rrp'] < 0).sum()
print(f"\nNegative price intervals: {n_negative:,} ({n_negative/len(df)*100:.2f}%)")

## 4. Graph 3: Weekday-Hour Heatmap

**Purpose**: Reveal typical price patterns (peak hours, weekend differences).

In [ ]:
# Mean price heatmap
fig = plots.plot_weekday_hour_heatmap(df, agg_func='mean')
plt.show()

In [ ]:
# Median price heatmap (less sensitive to spikes)
fig = plots.plot_weekday_hour_heatmap(df, agg_func='median', save_name='03b_weekday_hour_median')
plt.show()

## 5. Additional EDA

In [ ]:
# Add time features for analysis
df_features = features.compute_time_features(df)

# Price by hour
fig, ax = plt.subplots(figsize=(12, 5))
hourly = df_features.groupby('hour')['rrp'].agg(['mean', 'median', 'std'])
ax.plot(hourly.index, hourly['mean'], 'o-', label='Mean')
ax.plot(hourly.index, hourly['median'], 's-', label='Median')
ax.fill_between(hourly.index, 
                hourly['mean'] - hourly['std'],
                hourly['mean'] + hourly['std'],
                alpha=0.2)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('RRP ($/MWh)')
ax.set_title('Price by Hour of Day')
ax.legend()
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
# Price by month
fig, ax = plt.subplots(figsize=(12, 5))
monthly = df_features.groupby('month')['rrp'].agg(['mean', 'median', 'std'])
x = range(len(monthly))
ax.bar(x, monthly['mean'], yerr=monthly['std'], capsize=3, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'][:len(monthly)])
ax.set_xlabel('Month')
ax.set_ylabel('Mean RRP ($/MWh)')
ax.set_title('Mean Price by Month')
plt.tight_layout()
plt.show()

In [ ]:
# Demand analysis (if available)
if 'total_demand' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Demand distribution
    ax1 = axes[0]
    ax1.hist(df['total_demand'].dropna(), bins=50, color='orange', alpha=0.7)
    ax1.set_xlabel('Total Demand (MW)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Demand Distribution')
    
    # Price vs Demand scatter
    ax2 = axes[1]
    sample = df.sample(min(10000, len(df)), random_state=42)
    ax2.scatter(sample['total_demand'], sample['rrp'], alpha=0.3, s=5)
    ax2.set_xlabel('Total Demand (MW)')
    ax2.set_ylabel('RRP ($/MWh)')
    ax2.set_title('Price vs Demand')
    
    plt.tight_layout()
    plt.show()
else:
    print("Demand data not available")

## 6. Key Findings Summary

Record your observations here:

- **Price range**: _fill in_
- **Peak hours**: _fill in_
- **Weekend vs weekday**: _fill in_
- **Seasonal pattern**: _fill in_
- **Extreme events**: _fill in_

---
## Next Steps
Proceed to `03_spike_events.ipynb` for spike detection and driver analysis.